## Distributed Tracing with X-Ray

Welcome to the lesson on distributed tracing with AWS X-Ray. So far in this course, you have learned how to keep your AWS credentials secure, protect your APIs, and monitor your applications with CloudWatch. Now, we will focus on understanding how requests move through your application, especially when it uses multiple AWS services.

Distributed tracing helps you see the path a request takes as it travels through different parts of your system. This is important because modern cloud applications often use many services, and it can be hard to know where problems or slowdowns happen. AWS X-Ray is a tool that helps you trace these requests, find bottlenecks, and understand how your services work together.

Here's how the tracing flow works:

```text
┌─────────────────┐
│  Your Python    │
│  Application    │
└────────┬────────┘
         │
         │ (X-Ray SDK captures traces)
         ▼
┌─────────────────┐
│  Segment        │──┐
│  (Full trace)   │  │
└─────────────────┘  │
         │           │
         ├───────────┤
         ▼           ▼
    ┌────────┐  ┌────────┐
    │Subseg. │  │Subseg. │  (DynamoDB call, etc.)
    └────────┘  └────────┘
         │
         │ (Sent by X-Ray Daemon)
         ▼
┌─────────────────┐
│  AWS X-Ray      │
│  Service        │  (Store & visualize traces)
└─────────────────┘
```

By the end of this lesson, you will know how to add AWS X-Ray tracing to a Python application that uses AWS services, so you can see detailed information about what happens when your code runs.

---

## Quick Recall: Using AWS SDKs in Python

Before we dive into tracing, let's quickly remind ourselves how we use AWS SDKs in Python. In previous lessons, you used the `boto3` library to interact with AWS services like DynamoDB and CloudWatch.

For example, to connect to a DynamoDB table, you might write:

```python
import boto3

dynamodb = boto3.resource('dynamodb')
table = dynamodb.Table('Orders')
```

This code creates a DynamoDB resource and gets a reference to the `Orders` table. You can then use this table object to read or write data.

In this lesson, we will build on this by adding tracing to these operations.

---

## Setting Up AWS X-Ray in Your Python Application

To use AWS X-Ray in your Python application, you need to install and configure the AWS X-Ray SDK. On CodeSignal, the SDK is already installed for you, but it's good to know how to do this on your own machine.

You would normally install the SDK with:

```shell
pip install aws-xray-sdk
```

Next, you need to import and configure the X-Ray recorder in your Python code. Here's how you start:

```python
from aws_xray_sdk.core import xray_recorder, patch_all

xray_recorder.configure(service='OrdersWorker')
```

* `xray_recorder` is the main object you use to record traces.
* The `configure` method sets the name of your service. This helps you identify traces from this part of your application.

To make tracing work with AWS services like DynamoDB, you also need to patch the libraries you use. This is done with:

```python
patch_all()
```

* `patch_all()` automatically adds tracing to supported libraries, such as `boto3` and `requests`.

On CodeSignal, the X-Ray daemon (a background process that collects trace data) is already running for you. On your own machine, you would need to download and start the X-Ray daemon as well.

---

## Adding Tracing to Your Code

Now, let's see how to add tracing to your code step by step. We will use a simple example in which we write an item to a DynamoDB table and trace this operation.

### Step 1: Import and Configure X-Ray

First, import the necessary modules and configure X-Ray:

```python
from aws_xray_sdk.core import xray_recorder, patch_all
import boto3

xray_recorder.configure(service='OrdersWorker')
patch_all()
```

* This sets up X-Ray and ensures that all supported libraries are traced.

### Step 2: Connect to DynamoDB

Next, create a DynamoDB resource as you did before:

```python
dynamodb = boto3.resource('dynamodb')
```

### Step 3: Add a Traced Operation

Now, let's write a function that puts an item into the `Orders` table. When you patched libraries with `patch_all()`, X-Ray already started tracing DynamoDB calls automatically. However, you can create custom subsegments to make specific operations easier to find and analyze.

A subsegment is a part of a trace that shows what happens during a specific step. You should create custom subsegments when you want to:

* Label important operations so they're easy to spot in traces (like "process-payment" or "send-notification")
* Measure specific code blocks to see exactly how long they take
* Group related operations together, even if they involve multiple service calls

The automatic tracing from `patch_all()` will capture the low-level AWS SDK calls, but custom subsegments let you organize traces in a way that matches your business logic. This makes it much easier to understand what's happening when you review traces later.

Let's create a custom subsegment for our DynamoDB write:

```python
def work():
    table = dynamodb.Table('Orders')
    with xray_recorder.in_subsegment('dynamo-put'):
        table.put_item(Item={"order_id": "o-200", "amount": 77.7})
```

* `with xray_recorder.in_subsegment('dynamo-put'):` creates a subsegment named `dynamo-put`. Even though `patch_all()` will automatically trace the `put_item` call, this custom subsegment gives you a clear, labeled view of this specific operation in your traces.
* `table.put_item(...)` writes a new order to the table.

### Step 4: Run the Function and View Traces

Finally, call the function and print a message when done:

```python
if __name__ == "__main__":
    work()
    print("Done")
```

When you run this code, you should see:

```text
Done
```

The trace data is sent to the X-Ray daemon, which collects and sends it to AWS X-Ray. Now let's see how to view this data.

---

## Accessing the X-Ray Console

1. Open the AWS Management Console and navigate to the X-Ray service
2. In the left sidebar, click on **Traces** to see individual requests
3. Click on **Service map** to see how your services connect to each other

---

## Understanding Trace Data

When you view traces, you'll see:

* **Service Map**: A visual diagram showing your application (`OrdersWorker`) connected to DynamoDB. Each service is shown as a node, and connections show the requests between them.
* **Trace List**: A table showing individual requests with their duration and status. Each trace represents one execution of your code.
* **Trace Details**: Click on a specific trace to see a timeline breakdown. You'll see:
  * The main segment (your `OrdersWorker` service)
  * Your custom `dynamo-put` subsegment
  * The automatic DynamoDB call captured by `patch_all()`
  * Timing information showing how long each part took

> **Important timing note:** Traces may take 30-60 seconds to appear in the X-Ray console after your code runs. If you don't see your traces immediately:
>
> * Wait a minute and refresh the page
> * Check the time filter in the top-right corner - make sure it includes the last few minutes (e.g., "Last 5 minutes")
> * Look for traces with your service name `OrdersWorker`
>
> For example, if your trace took 150ms total, you might see the `dynamo-put` subsegment took 145ms, showing that most of the time was spent on the DynamoDB write.

---

## Review and What's Next

In this lesson, you learned how to:

* Set up AWS X-Ray in a Python application.
* Patch libraries like `boto3` to automatically trace AWS service calls.
* Use subsegments to trace specific operations, such as writing to DynamoDB.

You saw how to build up the code step by step, and how each part helps you trace what your application is doing. In the practice exercises, you will get hands-on experience adding tracing to your own code and exploring the trace data.

Congratulations on reaching the end of this course! You now have a strong foundation in developer security and observability on AWS. Keep practicing and applying these skills to build secure, reliable, and observable cloud applications.

## Configure Your X-Ray Service

Now that you understand how AWS X-Ray works, it's time to put your knowledge into practice! You have been given a partially working script that writes an order to a DynamoDB table, but the X-Ray setup is incomplete.

Your objective is to complete the configuration by adding the missing piece:

* Configure the X-Ray recorder with the service name `"OrderProcessor"` and `sampling=False` using `xray_recorder.configure(...)`.

Look for the TODO comment in the code — it will guide you to exactly where you need to add the missing configuration. Keep the rest of the starter code as-is: `patch_all()` still patches the AWS SDK libraries so calls are automatically traced, and the `with xray_recorder.in_segment('main'):` block must stay in place, since a subsegment needs an active segment to work. Once you complete this step, running the script will write an order to the `Orders` table inside a custom `dynamo-put` subsegment, so the operation shows up clearly in your X-Ray traces.

```python
from aws_xray_sdk.core import xray_recorder, patch_all
import boto3

# TODO: Configure the X-Ray recorder with service='OrderProcessor' and sampling=False

patch_all()

dynamodb = boto3.resource('dynamodb')

def work():
    table = dynamodb.Table('Orders')
    with xray_recorder.in_segment('main'):
        with xray_recorder.in_subsegment('dynamo-put'):
            table.put_item(Item={"order_id": "o-200", "amount": 77.7})

if __name__ == "__main__":
    work()
    print("Done")
```

Here is the completed script with the X-Ray recorder configured:

```python
from aws_xray_sdk.core import xray_recorder, patch_all
import boto3

xray_recorder.configure(service='OrderProcessor', sampling=False)

patch_all()

dynamodb = boto3.resource('dynamodb')

def work():
    table = dynamodb.Table('Orders')
    with xray_recorder.in_segment('main'):
        with xray_recorder.in_subsegment('dynamo-put'):
            table.put_item(Item={"order_id": "o-200", "amount": 77.7})

if __name__ == "__main__":
    work()
    print("Done")
```

`xray_recorder.configure(service='OrderProcessor', sampling=False)` names the service so its traces are easy to identify in the X-Ray console, and `sampling=False` disables sampling so every request is traced instead of only a subset — useful while you're developing and want to see every trace. `patch_all()` instruments `boto3` (and other supported libraries) so the `put_item` call is captured automatically, and the outer `with xray_recorder.in_segment('main'):` provides the active segment that the `dynamo-put` subsegment attaches to.

## Enable Automatic AWS SDK Tracing

Excellent work on setting up your X-Ray service configuration! Now, let's take the next step and enable automatic tracing for all your AWS operations.

You have a Python script with X-Ray properly configured, but right now, it only traces operations that you manually wrap in subsegments. X-Ray can do much more than that — it can automatically trace every AWS SDK call your application makes without you having to wrap each one individually.

Your task is to enable this automatic tracing by adding the patch_all() function call. This single line tells X-Ray to automatically instrument supported libraries like boto3, so all your DynamoDB operations (and other AWS service calls) will be traced automatically.

Look for the TODO comment that shows you where to add this line. The patching should happen after your X-Ray configuration but before you start using AWS services.

Once you add this function call, you'll see much richer trace data that includes both your manual subsegments and automatic traces of every AWS operation your code performs.


```python
from aws_xray_sdk.core import xray_recorder, patch_all
from decimal import Decimal
import boto3

xray_recorder.configure(service='OrderProcessor', sampling=False)
# TODO: Add the patch_all() function call to enable automatic tracing of AWS SDK operations

dynamodb = boto3.resource('dynamodb')

def work():
    table = dynamodb.Table('Orders')
    with xray_recorder.in_subsegment('dynamo-put'):
        table.put_item(Item={"order_id":"o-300","amount":Decimal('89.5')})

if __name__ == "__main__":
    xray_recorder.begin_segment('main')
    try:
        work()
        print("Done")
    finally:
        xray_recorder.end_segment()
```

Here is the completed script with automatic AWS SDK tracing enabled:

```python
from aws_xray_sdk.core import xray_recorder, patch_all
from decimal import Decimal
import boto3

xray_recorder.configure(service='OrderProcessor', sampling=False)
patch_all()

dynamodb = boto3.resource('dynamodb')

def work():
    table = dynamodb.Table('Orders')
    with xray_recorder.in_subsegment('dynamo-put'):
        table.put_item(Item={"order_id":"o-300","amount":Decimal('89.5')})

if __name__ == "__main__":
    xray_recorder.begin_segment('main')
    try:
        work()
        print("Done")
    finally:
        xray_recorder.end_segment()
```

`patch_all()` is added right after `xray_recorder.configure(...)` and before the `dynamodb` resource is created, so every supported library — including `boto3` — is instrumented before it's used. With patching enabled, the low-level `put_item` call made inside `table.put_item(...)` is now automatically captured by X-Ray, in addition to the manual `dynamo-put` subsegment, giving you a complete, layered view of the operation in your traces.

## Add Custom Subsegment Tracing

Perfect! You now have automatic tracing enabled for all your AWS operations. Let's dive deeper into creating custom subsegments to gain even more detailed insights into your application's performance.

While automatic tracing shows you when AWS services are called, sometimes you want to focus on specific operations within your code. Custom subsegments let you isolate particular steps and see exactly how long they take to complete.

Your task is to wrap the DynamoDB put_item operation with a custom subsegment named 'store-order'. This will create a dedicated trace section that shows the performance of just this database write operation.

Look for the TODO comment in the work() function, and add the subsegment context manager around the table.put_item() call. This will help you identify any bottlenecks in your database operations and give you precise timing data for this critical step.


```python
from aws_xray_sdk.core import xray_recorder, patch_all
import boto3
from decimal import Decimal

xray_recorder.configure(
    service='OrderProcessor',
    sampling=False  # Disable dynamic sampling
)
patch_all()  # patches boto3, requests, etc.

dynamodb = boto3.resource('dynamodb')

def work():
    table = dynamodb.Table('Orders')
    # TODO: Wrap the put_item operation below with a subsegment named 'store-order'
    try:
        table.put_item(Item={"order_id": "o-400", "amount": Decimal('125.50')})
    except Exception:
        # In demo environment, table operations may not be available
        pass

if __name__ == "__main__":
    xray_recorder.begin_segment('OrderProcessor')
    try:
        work()
        print("Done")
    finally:
        xray_recorder.end_segment()
```

Here is the completed `work()` function with the `store-order` subsegment added:

```python
from aws_xray_sdk.core import xray_recorder, patch_all
import boto3
from decimal import Decimal

xray_recorder.configure(
    service='OrderProcessor',
    sampling=False  # Disable dynamic sampling
)
patch_all()  # patches boto3, requests, etc.

dynamodb = boto3.resource('dynamodb')

def work():
    table = dynamodb.Table('Orders')
    with xray_recorder.in_subsegment('store-order'):
        try:
            table.put_item(Item={"order_id": "o-400", "amount": Decimal('125.50')})
        except Exception:
            # In demo environment, table operations may not be available
            pass

if __name__ == "__main__":
    xray_recorder.begin_segment('OrderProcessor')
    try:
        work()
        print("Done")
    finally:
        xray_recorder.end_segment()
```

The `with xray_recorder.in_subsegment('store-order'):` block wraps the entire `try/except` around `table.put_item(...)`, so the subsegment's timing covers the full attempt at the database write — including the case where it fails and is caught by the `except` clause. This gives you a dedicated `store-order` section in your trace, letting you see precisely how long this specific operation takes, separate from the automatic tracing that `patch_all()` already provides for the underlying `boto3` call.

## Trace Multiple Operations with Subsegments

Nice work mastering custom subsegments for individual operations! Now, let's advance your tracing skills by learning how to break down complex functions into multiple detailed segments.

In real applications, a single function often performs several different operations, and you want to see exactly which step is taking the most time or causing issues. Instead of tracing the entire function as one big block, you can create separate subsegments for each major operation.

Your task is to add two different named subsegments to trace the distinct operations in the process_order() function:

    Wrap the table reference operation with a subsegment named 'get-table-reference'.
    Wrap the data writing operation with a subsegment named 'write-item'.

Look for the TODO comments that show you exactly where to add each subsegment wrapper. Each subsegment should have a clear, descriptive name that makes it easy to understand what operation it's tracking.

This approach helps you pinpoint exactly which part of your function might be slow, making performance optimization much more targeted and effective.

```python
from aws_xray_sdk.core import xray_recorder, patch_all
import boto3

xray_recorder.configure(service='OrderProcessor')
patch_all()

dynamodb = boto3.resource('dynamodb')

def process_order():
    # TODO: Wrap the table reference operation below with a subsegment named 'get-table-reference'
    table = dynamodb.Table('Orders')
    
    # TODO: Wrap the put_item operation below with a subsegment named 'write-item'
    table.put_item(Item={"order_id": "o-500", "amount": 199.99, "status": "pending"})

if __name__ == "__main__":
    process_order()
    print("Done")
```

Here is the completed `process_order()` function with both subsegments added:

```python
from aws_xray_sdk.core import xray_recorder, patch_all
import boto3

xray_recorder.configure(service='OrderProcessor')
patch_all()

dynamodb = boto3.resource('dynamodb')

def process_order():
    with xray_recorder.in_subsegment('get-table-reference'):
        table = dynamodb.Table('Orders')

    with xray_recorder.in_subsegment('write-item'):
        table.put_item(Item={"order_id": "o-500", "amount": 199.99, "status": "pending"})

if __name__ == "__main__":
    process_order()
    print("Done")
```

Splitting the function into two subsegments — `get-table-reference` and `write-item` — lets you see each step as its own labeled section in the trace timeline. If a trace ever shows a slowdown, you can immediately tell whether the delay came from resolving the table reference or from the actual write to DynamoDB, instead of having to guess which part of `process_order()` was responsible.

## Debug Broken X-Ray Integration

Outstanding work building up your X-Ray tracing skills through each step! Now it's time to bring everything together and test your ability to debug real X-Ray integration problems.

You have a Python script that's supposed to trace multiple DynamoDB operations, but it contains several common X-Ray setup mistakes that are preventing tracing from working properly. This is your chance to apply all the concepts you've learned and fix a realistic, broken implementation.

Your task is to identify and fix all the X-Ray integration issues in this code:

* Missing imports for essential X-Ray modules
* Incorrect configuration timing and setup order
* Missing function calls for automatic tracing
* Broken subsegments that don't properly wrap their operations

Some problems have TODO comments to guide you, but others look like normal code that you'll need to debug yourself. The script should perform multiple DynamoDB operations while properly tracing each one with both automatic instrumentation and custom subsegments.

When you fix all the issues, you'll have a fully functional X-Ray setup that demonstrates mastery of distributed tracing concepts and prepares you to debug similar problems in real applications.

```python
from aws_xray_sdk.core import xray_recorder
from decimal import Decimal
import boto3

# TODO: Add the missing import needed for automatic AWS SDK tracing

dynamodb = boto3.resource('dynamodb')

# TODO: Configure the X-Ray recorder with service name 'OrderProcessor'

def process_orders():
    # TODO: Add the missing function call to enable automatic tracing of AWS operations
    
    with xray_recorder.in_subsegment('get-table-reference'):
        table = dynamodb.Table('Orders')
    
    # First order
    with xray_recorder.in_subsegment('create-order-1'):
        table.put_item(Item={"order_id": "o-100", "amount": Decimal('45.99'), "status": "pending"})
    
    # Second order  
    table.put_item(Item={"order_id": "o-101", "amount": Decimal('78.50'), "status": "confirmed"})
    with xray_recorder.in_subsegment('create-order-2'):
        pass

def check_orders():
    table = dynamodb.Table('Orders')
    with xray_recorder.in_subsegment('scan-orders'):
        response = table.scan(Limit=5)
    return len(response.get('Items', []))

if __name__ == "__main__":
    with xray_recorder.in_segment('main'):
        process_orders()
        count = check_orders()
        print(f"Processed orders, found {count} items")
    print("Done")
```

Here is the fully debugged script:

```python
from aws_xray_sdk.core import xray_recorder, patch_all
from decimal import Decimal
import boto3

# Configure X-Ray and patch the AWS SDK BEFORE any boto3 resource is created
xray_recorder.configure(service='OrderProcessor')
patch_all()

dynamodb = boto3.resource('dynamodb')

def process_orders():
    with xray_recorder.in_subsegment('get-table-reference'):
        table = dynamodb.Table('Orders')

    # First order
    with xray_recorder.in_subsegment('create-order-1'):
        table.put_item(Item={"order_id": "o-100", "amount": Decimal('45.99'), "status": "pending"})

    # Second order
    with xray_recorder.in_subsegment('create-order-2'):
        table.put_item(Item={"order_id": "o-101", "amount": Decimal('78.50'), "status": "confirmed"})

def check_orders():
    table = dynamodb.Table('Orders')
    with xray_recorder.in_subsegment('scan-orders'):
        response = table.scan(Limit=5)
    return len(response.get('Items', []))

if __name__ == "__main__":
    with xray_recorder.in_segment('main'):
        process_orders()
        count = check_orders()
        print(f"Processed orders, found {count} items")
    print("Done")
```

The four bugs and their fixes:

1. **Missing import** — `patch_all` was never imported. Change `from aws_xray_sdk.core import xray_recorder` to `from aws_xray_sdk.core import xray_recorder, patch_all`.
2. **Wrong setup order** — `xray_recorder.configure(...)` was missing entirely and its TODO sat *after* `dynamodb = boto3.resource('dynamodb')`. Configuration must run before any AWS resource is created, so `xray_recorder.configure(service='OrderProcessor')` moves to the top of the module.
3. **`patch_all()` in the wrong place** — the TODO put it *inside* `process_orders()`, but by then `dynamodb` has already been created unpatched, and it would re-patch on every call. `patch_all()` belongs at module level, right after `configure(...)` and before the `boto3.resource(...)` line.
4. **Broken `create-order-2` subsegment** — the second `put_item` ran *outside* the subsegment, which only wrapped an empty `pass`. Move the `put_item` call inside the `with xray_recorder.in_subsegment('create-order-2'):` block so the subsegment actually times the write.